In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Atténuation d'erreurs par réseau de tenseurs (TEM) : une fonction Qiskit par Algorithmiq
*Consulte la [référence API](https://docs.quantum.ibm.com/api/functions/algorithmiq-tem)*

> **Note:** Les fonctions Qiskit sont une fonctionnalité expérimentale disponible uniquement pour les utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et susceptibles de changer.



<Accordion>
<AccordionItem title="Package versions">

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>

## Vue d'ensemble
La méthode d'atténuation d'erreurs par réseau de tenseurs (TEM) d'Algorithmiq est un algorithme hybride quantique-classique conçu pour effectuer l'atténuation du bruit entièrement lors de l'étape de post-traitement classique. Avec TEM, l'utilisateur peut calculer les valeurs attendues des observables en atténuant les erreurs inévitables induites par le bruit qui se produisent sur le matériel quantique, avec une précision accrue et une meilleure efficacité en termes de coûts, ce qui en fait une option très attractive pour les chercheurs en informatique quantique et les praticiens industriels.

La méthode consiste à construire un réseau de tenseurs représentant l'inverse du canal de bruit global affectant l'état du processeur quantique, puis à appliquer la carte aux résultats de mesure informationnellement complets acquis de l'état bruité pour obtenir des estimateurs non biaisés des observables.

Comme avantage, TEM exploite des mesures informationnellement complètes pour donner accès à un vaste ensemble de valeurs attendues atténuées des observables et a une surcharge d'échantillonnage optimale sur le matériel quantique, comme décrit dans Filippov et al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), et Filippov et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). La surcharge de mesure fait référence au nombre de mesures supplémentaires nécessaires pour effectuer une atténuation d'erreurs efficace, un facteur critique dans la faisabilité des calculs quantiques. Par conséquent, TEM a le potentiel de permettre l'avantage quantique dans des scénarios complexes, comme les applications dans les domaines du chaos quantique, de la physique à plusieurs corps, de la dynamique de Hubbard et des simulations de chimie de petites molécules.

Les principales caractéristiques et avantages de TEM peuvent être résumés comme suit :

1. **Surcharge de mesure optimale** : TEM est optimal par rapport aux [bornes théoriques](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601), ce qui signifie qu'aucune méthode ne peut atteindre une surcharge de mesure plus faible. En d'autres termes, TEM nécessite le nombre minimal de mesures supplémentaires pour effectuer l'atténuation d'erreurs. Cela signifie à son tour que TEM utilise un temps d'exécution quantique minimal.
2. **Rentabilité** : Comme TEM gère l'atténuation du bruit entièrement dans l'étape de post-traitement, il n'est pas nécessaire d'ajouter des circuits supplémentaires à l'ordinateur quantique, ce qui rend non seulement le calcul moins coûteux mais réduit également le risque d'introduire des erreurs supplémentaires dues aux imperfections des dispositifs quantiques.
3. **Estimation de plusieurs observables** : Grâce aux mesures informationnellement complètes, TEM estime efficacement plusieurs observables avec les mêmes données de mesure de l'ordinateur quantique.
4. **Atténuation des erreurs de mesure** : La fonction Qiskit TEM inclut également une [méthode propriétaire d'atténuation des erreurs de mesure](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154) capable de réduire significativement les erreurs de lecture après une courte exécution de calibration.
5. **Précision** : TEM améliore significativement la précision et la fiabilité des simulations numériques quantiques, rendant les algorithmes quantiques plus précis et fiables.

## Description
La fonction TEM te permet d'obtenir des valeurs attendues atténuées des erreurs pour plusieurs observables sur un circuit quantique avec une surcharge d'échantillonnage minimale. Le circuit est mesuré avec une mesure à opérateur-valeur positif informationnellement complète (IC-POVM), et les résultats de mesure collectés sont traités sur un ordinateur classique. Cette mesure est utilisée pour effectuer les méthodes de réseau de tenseurs et construire une carte d'inversion du bruit. La fonction applique une carte qui inverse complètement le circuit bruité en utilisant des réseaux de tenseurs pour représenter les couches bruitées.

![Schéma TEM](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Estimation atténuée des erreurs d'une observable O via le post-traitement des résultats de mesure du processeur quantique bruité. U et N désignent une opération quantique idéale et la carte de bruit associée. D représente un tenseur d'opérateurs duaux aux effets dans la mesure IC. Le module d'atténuation du bruit M est un réseau de tenseurs contracté efficacement de l'intérieur vers l'extérieur.")

Une fois les circuits soumis à la fonction, ils sont transpilés et optimisés pour minimiser le nombre de couches avec des portes à deux qubits (les portes les plus bruitées sur les dispositifs quantiques). Le bruit affectant les couches est appris via [Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner) en utilisant un modèle de bruit de Pauli-Lindblad sparse tel que décrit dans E. van den Berg, Z. Minev, A. Kandala, K. Temme, Nat. Phys. (2023). [arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Le modèle de bruit est une description précise du bruit sur le dispositif, capable de capturer des caractéristiques subtiles, y compris la diaphonie entre qubits. Cependant, le bruit sur les dispositifs peut fluctuer et dériver, et le bruit appris pourrait ne pas être précis au moment où l'estimation est effectuée. Cela peut entraîner des résultats inexacts.

## Premiers pas
Authentifie-toi en utilisant ta [clé API IBM Quantum Platform](http://quantum.cloud.ibm.com/), et sélectionne la fonction TEM comme suit. (Cet extrait suppose que tu as déjà [enregistré ton compte](/guides/functions#install-qiskit-functions-catalog-client) dans ton environnement local.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Exemple
L'extrait suivant montre un exemple où TEM est utilisé pour calculer les valeurs attendues d'une observable pour un circuit quantique simple.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device
# reported by IBM if not specified
backend_name = "ibm_marrakesh"

# Run the TEM function (uses around three minutes of QPU time)
job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Utilise les APIs Qiskit Serverless pour vérifier le statut de ta charge de travail Qiskit Function :

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs
print(evs[0])

0.02165380888171687


Tu peux récupérer les résultats comme suit :